In [1]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

from examples.seismic import Model, plot_velocity, TimeAxis, RickerSource, Receiver
from devito import TimeFunction, VectorTimeFunction, Eq, solve, Operator
from devito.finite_differences.operators import div, grad
from matplotlib.animation import FuncAnimation
from devito import NODE

from lista_utils import *

# QUESTÃO 1

In [ ]:
# Construindo modelo de velocidade

nx, nz = 2001, 151 # Quantidade de pontos nas direções X e Z (151 pontos)
dx, dz = 10, 10 # Espaçamento entre pontos nas direções X e Z (10 metros)
origin = (0., 0.)  # Coordenadas da origem do modelo
dtype = 'float32'
nbl = 100
space_order = 8

vp = np.ones((nx,nz), dtype=dtype) * 1.5
vp[:, nz // 6 :] = 6
rho = vp
b = 1 / rho

model = Model(vp=vp, b=b, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs='damp')

# Plotando o modelo de velocidade

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet'}

fig, ax = plt.subplots(figsize=(6,4))

img = ax.imshow(model.vp.data.T, **plot_options)
ax.set_title('Modelo Vp')
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Profundidade (m)')

cbar = fig.colorbar(img)
cbar.set_label('Vp (km/s)')

fig.tight_layout()
plt.show()

In [ ]:
# Definindo a fonte sísmica (Ricker)

t0 = 0.  # Tempo inicial da modelagem t=0ms
tn = 6000.  # Tempo final da modelagem t=1000ms
dt = model.critical_dt  # Tempo entre iterações (2ms)
f0 = 0.01  # Frequência de pico da wavelet (20Hz = 0.020 kHz)
ns = 1

time_range = TimeAxis(start=t0, stop=tn, step=dt)
src = RickerSource(name='src', grid=model.grid, f0=f0, npoint=ns, time_range=time_range)

# Definindo coordenadas da fonte
src.coordinates.data[0, 0] = model.domain_size[0] * .5 # Centralizando no eixo x
src.coordinates.data[0, 1] = 40 # Centralizando no eixo z

# Plotando a fonte e coordenadas
print(f'Coordenadas: \n\tx: {src.coordinates.data[0,0]}m \n\tz: {src.coordinates.data[0,1]}m')

fig, ax = plt.subplots(figsize=(7,2))

ax.plot(src.time_values, src.data)
ax.set_xlabel('Tempo (ms)')
ax.set_ylabel('Amplitude')

fig.tight_layout()
plt.show()

In [ ]:
# Definindo os receptores

ng = nx

rec = Receiver(name='rec', grid=model.grid, npoint=ng, time_range=time_range)

# Definindo coordenadas dos receptores
rec.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], ng) # Definindo coordenadas equiespaçadas em x
rec.coordinates.data[:, 1] = 0.  # Profundidade de 20m

# Plotting velocity model com fonte e receptores

plot_aquisition_setup(model, src, rec)

Partindo da equação da onda acústica de segunda ordem:

$$ \frac{1}{v_p^2} \frac{\partial^2 P}{\partial t^2} - \nabla^2 P = 0$$

In [ ]:
# Plotando sismogramas

P, rec = acoustic_forward(model, src, rec, order=2)

In [ ]:
amax = max(rec.data.max(), abs(rec.data.min())) * .5
plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'vmin':-amax, 'vmax':amax}

fig, ax = plt.subplots(figsize=(20,30))

ax.imshow(rec.data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

### a)

In [ ]:
# Plotando snapshots

plot_snaps(P, model, src, 100, 2000, 200, cols=6)

### b)

In [ ]:
# Plotando filmagem da propagação

# OBS01: ESTA CÉLULA DEMORA 2min PARA EXECUTAR
# OBS02: PARA EXECUTAR O VÍDEO, BASTA DAR PLAY NO WIDGET QUE APARECERÁ

output = plot_video(P, rec, model, interval=1)

output

### c)

A amplitude da onda diminui com o passar do tempo devido a dois principais fatores: divergência esférica e amortecimento da borda. O primeiro fator é um fator físico e acontece na realidade. A perda de energia por divergência esférica se dá devido à propagação radial/esférica da onda. A energia inicial da onda, concentrada em aproximadamente um ponto, é a única responsável por sua propagação (energia total) e à medida em que a onda se propaga essa mesma energia, antes concentrada em um ponto, deve se dividir em uma frente de onda esférica cujo raio aumenta com o tempo.

De acordo com a equação da área da superfície de uma esfera

$$ A=4 \pi R^2 $$

podemos dizer que a energia da onda diminui com o quadrado do raio.

Outro fator impactante para a diminuição da amplitude na modelagem é o amortecimento da borda. Em casos de simulação computacional, deve ser adicionada ao modelo uma borda atenuante para que a simulação se torne mais coerente com a realidade (modelo infinito), afinal na Terra não há bordas. Portanto, grande parte da atenuação da onda nessa modelagem se deu devido à borda atenuante.

### d)

\begin{cases}
    \rho \frac{\partial \mathbf{V}}{\partial t} - \nabla P = 0 \\ \\
    \frac{\partial P}{\partial t} - \kappa \nabla \cdot \mathbf{V} = \mathbf{f}
\end{cases}

In [ ]:
# Configurando aquisição

ng = nx

rec_vx = Receiver(name='rec_vx', grid=model.grid, npoint=ng, time_range=time_range)
rec_vz = Receiver(name='rec_vz', grid=model.grid, npoint=ng, time_range=time_range)
rec_p = Receiver(name='rec_p', grid=model.grid, npoint=ng, time_range=time_range)

# Prescribe even spacing for receivers along the x-axis
rec_vx.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=ng)
rec_vx.coordinates.data[:, 1] = 20.  # postion of receiver at 400 m depth for vx

rec_vz.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=ng)
rec_vz.coordinates.data[:, 1] = 20.  # postion of receiver at 400 m depth for vz

rec_p.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=ng)
rec_p.coordinates.data[:, 1] = 20. # postion of receiver at 10 m of depth for pressure field

plot_aquisition_setup(model, src, rec_p)

In [ ]:
# Plotando sismogramas

rec = [rec_vx, rec_vz, rec_p]

V, P, rec = acoustic_modelling(model, src, rec, order=1)

plot_options = {'aspect':'auto', 'cmap':'Greys'}

fig, axes = plt.subplots(1, 3, figsize=(12,7))

axes[0].imshow(rec[0].data, **plot_options)
axes[0].set_title('Receptores (Vx)')
axes[1].imshow(rec[1].data, **plot_options)
axes[1].set_title('Receptores (Vz)')
axes[2].imshow(rec[2].data, **plot_options)
axes[2].set_title('Receptores (Pressão)')

fig.tight_layout()
plt.show()

In [ ]:
# Plotando snapshots

plot_snaps(P, model, src, 100, 2000, 20, cols=6)

In [ ]:
# --------- FIM --------- #